##### Copyright 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Neural Machine Translation with Attention: Korean → English

This notebook builds a sequence-to-sequence (seq2seq) model with a
Bahdanau/Luong-style attention mechanism to translate **Korean → English**.
It is adapted from the official TensorFlow tutorial *"Neural machine
translation with attention"*, which originally implements Spanish → English
translation. The overall architecture — encoder, cross-attention layer,
decoder, and training/inference pipeline — is preserved; only the dataset,
text preprocessing, and evaluation sections have been redesigned for Korean.

<table>
<tr>
  <td>
   <img width=400 src="https://www.tensorflow.org/images/tutorials/transformer/RNN%2Battention-words-spa.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>An encoder/decoder connected by attention (illustrative diagram from the original tutorial).</th>
<tr>
</table>

While RNN + attention is somewhat outdated compared to Transformers, it
remains a great way to build intuition for how attention mechanisms work
before moving on to Transformer-based architectures.

This example assumes some knowledge of TensorFlow fundamentals below the
level of a Keras layer:
  * [Working with tensors](https://www.tensorflow.org/guide/tensor) directly
  * [Writing custom `keras.Model`s and `keras.layers`](https://www.tensorflow.org/guide/keras/custom_layers_and_models)

After training the model in this notebook, you will be able to input a
Korean sentence, such as "*오늘 날씨가 정말 좋네요.*", and get back an
English translation such as "*the weather is really nice today.*"

The resulting model is exportable as a `tf.saved_model`, so it can be used
in other TensorFlow environments.

The translation quality is reasonable for a toy example, but the generated
attention heatmap is perhaps more interesting: it shows which Korean tokens
the model was "looking at" while generating each English word.

Note: Training in this notebook is intentionally kept small so the whole
notebook can run in Google Colab in a reasonable amount of time. See the
**Future Improvements** section at the end for ideas on scaling this up.

## Setup

In [ ]:
!pip install -q "tensorflow-text>=2.11"
!pip install -q einops
!pip install -q datasets sacrebleu

# A CJK-capable font is required so Korean (Hangul) characters render
# correctly in matplotlib figures (e.g. the attention heatmaps below).
!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-nanum > /dev/null


In [ ]:
import re
import time
import pathlib
import typing
from typing import Any, Tuple

import numpy as np
import einops

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm

import tensorflow as tf
import tensorflow_text as tf_text

# ---------------------------------------------------------------------------
# Register the Korean font we installed above so Hangul renders correctly
# in every matplotlib plot in this notebook (loss curves, attention maps).
# ---------------------------------------------------------------------------
for _font_path in [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
]:
    try:
        fm.fontManager.addfont(_font_path)
    except Exception:
        pass

plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False


This tutorial uses a lot of low level API's where it's easy to get shapes wrong. This class is used to check shapes throughout the tutorial.


In [ ]:
#@title
class ShapeChecker():
  def __init__(self):
    # Keep a cache of every axis-name seen
    self.shapes = {}

  def __call__(self, tensor, names, broadcast=False):
    if not tf.executing_eagerly():
      return

    parsed = einops.parse_shape(tensor, names)

    for name, new_dim in parsed.items():
      old_dim = self.shapes.get(name, None)

      if (broadcast and new_dim == 1):
        continue

      if old_dim is None:
        # If the axis name is new, add its length to the cache.
        self.shapes[name] = new_dim
        continue

      if new_dim != old_dim:
        raise ValueError(f"Shape mismatch for dimension: '{name}'\n"
                         f"    found: {new_dim}\n"
                         f"    expected: {old_dim}\n")

## The data

This tutorial uses a **Korean–English parallel corpus**. We primarily use
a subset of the [OPUS-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100)
`en-ko` dataset, downloaded through the Hugging Face `datasets` library. OPUS-100
is a large, publicly available, multilingual corpus built from the
[OPUS](https://opus.nlpl.eu/) collection of translated texts, and is one of
the datasets recommended for Korean–English machine translation research.

If the download is unavailable (for example, no internet access in the
runtime), the notebook automatically falls back to a small, hand-curated
sample corpus bundled directly in this notebook, so every cell below still
runs end-to-end.

Each sentence pair is stored one per line in the format:

```
안녕하세요.	Hello.
감사합니다.	Thank you.
오늘 날씨가 좋네요.	The weather is nice today.
```

i.e. **Korean\tEnglish**, encoded in UTF-8.

### Download and prepare the dataset

Steps taken to prepare the data:

1. Download (or fall back to a bundled sample of) Korean–English sentence
   pairs and save them locally as `kor_eng.txt`.
2. Load the pairs into NumPy arrays of raw strings.
3. Add `[START]` and `[END]` tokens to each sentence (done later, inside the
   text-standardization function).
4. Build a Korean vocabulary (context) and an English vocabulary (target)
   using `TextVectorization`.
5. Pad each sentence to a common length within each batch.

The notebook does **not** hardcode any absolute file paths — everything is
written to and read from the current working directory, so it works the
same way locally or in Google Colab.

In [ ]:
# ---------------------------------------------------------------------------
# Korean-English Parallel Corpus
# ---------------------------------------------------------------------------
# We try to download a subset of the OPUS-100 (en-ko) corpus from the
# Hugging Face `datasets` hub. If that fails for any reason (no internet
# access, hub unavailable, etc.), we fall back to a small, hand-curated
# sample corpus bundled directly below so the rest of the notebook still
# runs end-to-end.
# ---------------------------------------------------------------------------
NUM_EXAMPLES = 20000          # number of sentence pairs to keep (kept small for a quick tutorial run)
MAX_SENTENCE_CHARS = 60       # skip very long sentences to keep training fast

path_to_file = pathlib.Path('kor_eng.txt')

# A small, hand-curated fallback corpus (Korean, English), used only if the
# OPUS-100 download fails (e.g. no internet connection in the runtime).
SAMPLE_PAIRS = [('안녕하세요.', 'Hello.'), ('안녕히 가세요.', 'Goodbye.'), ('감사합니다.', 'Thank you.'), ('천만에요.', "You're welcome."), ('죄송합니다.', "I'm sorry."), ('실례합니다.', 'Excuse me.'), ('만나서 반갑습니다.', 'Nice to meet you.'), ('잘 지내세요?', 'How are you?'), ('저는 잘 지내고 있어요.', 'I am doing well.'), ('이름이 뭐예요?', 'What is your name?'), ('제 이름은 민수입니다.', 'My name is Minsu.'), ('몇 살이에요?', 'How old are you?'), ('저는 스무 살입니다.', 'I am twenty years old.'), ('어디에서 오셨어요?', 'Where are you from?'), ('저는 한국에서 왔습니다.', 'I am from Korea.'), ('오늘 날씨가 좋네요.', 'The weather is nice today.'), ('오늘 날씨가 정말 좋네요.', 'The weather is really nice today.'), ('비가 올 것 같아요.', "It looks like it's going to rain."), ('오늘은 너무 덥습니다.', 'It is too hot today.'), ('겨울에는 눈이 많이 옵니다.', 'It snows a lot in winter.'), ('내일은 흐릴 예정입니다.', 'Tomorrow will be cloudy.'), ('바람이 많이 붑니다.', 'The wind is blowing hard.'), ('오늘 아침은 쌀쌀합니다.', 'It is chilly this morning.'), ('봄이 되면 꽃이 핍니다.', 'Flowers bloom when spring comes.'), ('여름에는 비가 자주 옵니다.', 'It rains often in summer.'), ('저는 매일 아침 일곱 시에 일어납니다.', 'I wake up at seven every morning.'), ('저는 학교에 갑니다.', 'I go to school.'), ('그는 회사에서 일합니다.', 'He works at a company.'), ('우리는 집에서 저녁을 먹습니다.', 'We eat dinner at home.'), ('저는 커피를 좋아합니다.', 'I like coffee.'), ('그녀는 책 읽는 것을 좋아합니다.', 'She likes reading books.'), ('저는 매일 운동을 합니다.', 'I exercise every day.'), ('오늘 저녁에 친구를 만날 거예요.', 'I am going to meet a friend this evening.'), ('주말에는 보통 집에서 쉽니다.', 'I usually rest at home on weekends.'), ('저는 아침에 신문을 읽습니다.', 'I read the newspaper in the morning.'), ('이 음식은 정말 맛있습니다.', 'This food is really delicious.'), ('저는 김치를 매우 좋아합니다.', 'I like kimchi very much.'), ('저녁에 무엇을 먹고 싶어요?', 'What do you want to eat for dinner?'), ('이 식당은 유명합니다.', 'This restaurant is famous.'), ('저는 매운 음식을 잘 못 먹습니다.', "I can't eat spicy food well."), ('밥을 먹었어요?', 'Have you eaten?'), ('저는 아직 점심을 안 먹었어요.', "I haven't had lunch yet."), ('이 커피는 너무 답니다.', 'This coffee is too sweet.'), ('그는 요리를 잘합니다.', 'He cooks well.'), ('우리는 함께 저녁을 먹었습니다.', 'We had dinner together.'), ('저는 여행을 좋아합니다.', 'I like traveling.'), ('다음 주에 부산에 갈 거예요.', 'I am going to Busan next week.'), ('공항까지 얼마나 걸려요?', 'How long does it take to the airport?'), ('저는 처음으로 서울에 왔습니다.', 'This is my first time coming to Seoul.'), ('기차표를 예매했어요.', 'I booked a train ticket.'), ('이 지역은 관광지로 유명합니다.', 'This area is famous as a tourist spot.'), ('저는 제주도에 가본 적이 있습니다.', 'I have been to Jeju Island.'), ('호텔은 어디에 있어요?', 'Where is the hotel?'), ('여권을 보여 주세요.', 'Please show me your passport.'), ('우리는 비행기를 타고 여행했습니다.', 'We traveled by airplane.'), ('저는 인공지능을 공부하고 있습니다.', 'I am studying artificial intelligence.'), ('그는 대학교에서 컴퓨터 공학을 전공했습니다.', 'He majored in computer engineering at university.'), ('저는 회의에 참석해야 합니다.', 'I have to attend a meeting.'), ('이 프로젝트는 다음 달에 끝납니다.', 'This project ends next month.'), ('저는 매일 컴퓨터로 일합니다.', 'I work with a computer every day.'), ('그녀는 열심히 공부합니다.', 'She studies hard.'), ('저는 새로운 언어를 배우고 있습니다.', 'I am learning a new language.'), ('우리 팀은 새로운 프로그램을 개발하고 있습니다.', 'Our team is developing a new program.'), ('저는 오늘 시험이 있습니다.', 'I have an exam today.'), ('그는 대학원에서 인공지능을 연구합니다.', 'He researches artificial intelligence in graduate school.'), ('저는 세 형제가 있습니다.', 'I have three siblings.'), ('제 아버지는 의사입니다.', 'My father is a doctor.'), ('제 어머니는 선생님입니다.', 'My mother is a teacher.'), ('저는 동생이 한 명 있습니다.', 'I have one younger sibling.'), ('우리 가족은 함께 여행을 갑니다.', 'Our family goes on trips together.'), ('그는 제 친한 친구입니다.', 'He is my close friend.'), ('저는 할머니를 자주 찾아뵙니다.', 'I often visit my grandmother.'), ('그녀는 저의 동료입니다.', 'She is my colleague.'), ('저는 오늘 부모님을 뵈러 갑니다.', 'I am going to see my parents today.'), ('우리는 오랜 친구입니다.', 'We are old friends.'), ('이 스마트폰은 아주 빠릅니다.', 'This smartphone is very fast.'), ('인공지능 기술이 빠르게 발전하고 있습니다.', 'Artificial intelligence technology is developing rapidly.'), ('저는 매일 인터넷을 사용합니다.', 'I use the internet every day.'), ('이 프로그램은 사용하기 쉽습니다.', 'This program is easy to use.'), ('새로운 노트북을 샀습니다.', 'I bought a new laptop.'), ('이 앱은 매우 유용합니다.', 'This app is very useful.'), ('그는 소프트웨어 개발자입니다.', 'He is a software developer.'), ('데이터는 클라우드에 저장됩니다.', 'The data is stored in the cloud.'), ('저는 매일 이메일을 확인합니다.', 'I check my email every day.'), ('인공지능은 우리의 삶을 바꾸고 있습니다.', 'Artificial intelligence is changing our lives.'), ('저는 오늘 기분이 좋습니다.', 'I feel good today.'), ('그는 매우 행복해 보입니다.', 'He looks very happy.'), ('저는 조금 피곤합니다.', 'I am a little tired.'), ('그녀는 걱정이 많습니다.', 'She worries a lot.'), ('저는 이 영화가 정말 재미있었습니다.', 'I found this movie really fun.'), ('그것은 좋은 생각인 것 같습니다.', 'That seems like a good idea.'), ('저는 그 소식을 듣고 놀랐습니다.', 'I was surprised to hear the news.'), ('저는 당신의 의견에 동의합니다.', 'I agree with your opinion.'), ('그는 항상 긍정적입니다.', 'He is always positive.'), ('저는 오늘 하루가 정말 즐거웠습니다.', 'I really enjoyed today.'), ('이것은 얼마예요?', 'How much is this?'), ('화장실이 어디에 있어요?', 'Where is the restroom?'), ('조금만 기다려 주세요.', 'Please wait a moment.'), ('문을 좀 열어 주시겠어요?', 'Could you please open the door?'), ('다시 한 번 말씀해 주시겠어요?', 'Could you say that again?'), ('여기에 앉아도 될까요?', 'May I sit here?'), ('창문을 닫아 주세요.', 'Please close the window.'), ('도와주셔서 감사합니다.', 'Thank you for helping me.'), ('천천히 말씀해 주세요.', 'Please speak slowly.'), ('이 근처에 은행이 있어요?', 'Is there a bank near here?'), ('시간이 얼마나 걸려요?', 'How long does it take?'), ('저는 음악 듣는 것을 좋아합니다.', 'I like listening to music.'), ('그 영화는 아직 안 봤어요.', "I haven't seen that movie yet."), ('오늘은 제 생일입니다.', 'Today is my birthday.'), ('우리는 내일 만날 예정입니다.', 'We plan to meet tomorrow.'), ('이 책은 매우 유익합니다.', 'This book is very informative.'), ('저는 운전하는 것을 좋아하지 않습니다.', "I don't like driving."), ('그는 매일 아침 커피를 마십니다.', 'He drinks coffee every morning.'), ('오늘 회의는 취소되었습니다.', "Today's meeting was canceled."), ('저는 그 문제를 아직 해결하지 못했습니다.', "I haven't solved that problem yet."), ('이 도시는 매우 아름답습니다.', 'This city is very beautiful.'), ('저는 새로운 취미를 시작했습니다.', 'I started a new hobby.'), ('우리는 함께 영화를 보러 갔습니다.', 'We went to watch a movie together.'), ('그는 자주 늦게 일어납니다.', 'He often wakes up late.'), ('저는 매일 일기를 씁니다.', 'I write a diary every day.'), ('이 옷은 저에게 너무 큽니다.', 'This clothing is too big for me.'), ('그녀는 노래를 매우 잘 부릅니다.', 'She sings very well.'), ('저는 이번 주말에 이사합니다.', 'I am moving this weekend.'), ('우리는 새로운 계획을 세웠습니다.', 'We made a new plan.'), ('그는 회의에 늦었습니다.', 'He was late for the meeting.'), ('저는 아직 그 소설을 다 읽지 못했습니다.', "I haven't finished reading that novel yet."), ('이 길은 매우 복잡합니다.', 'This road is very busy.'), ('저는 매일 일곱 시간 동안 잡니다.', 'I sleep for seven hours every day.'), ('그녀는 세 개의 언어를 할 수 있습니다.', 'She can speak three languages.'), ('저는 오늘 중요한 발표가 있습니다.', 'I have an important presentation today.'), ('이 문제는 정말 어렵습니다.', 'This problem is really difficult.'), ('저는 어제 그를 만났습니다.', 'I met him yesterday.'), ('그들은 내년에 결혼할 예정입니다.', 'They plan to get married next year.'), ('저는 이 회사에서 오 년 동안 일했습니다.', 'I have worked at this company for five years.'), ('그는 매일 신문을 읽습니다.', 'He reads the newspaper every day.'), ('저는 오늘 아침에 늦게 일어났습니다.', 'I woke up late this morning.'), ('우리는 지난주에 산에 올라갔습니다.', 'We climbed the mountain last week.'), ('이 상품은 매우 인기가 많습니다.', 'This product is very popular.'), ('저는 아직 그 이메일에 답장하지 못했습니다.', "I haven't replied to that email yet."), ('그녀는 매우 친절합니다.', 'She is very kind.')]


def download_opus100(num_examples=NUM_EXAMPLES):
    """Download a subset of OPUS-100 (en-ko) via the Hugging Face `datasets` library."""
    from datasets import load_dataset

    print("Downloading OPUS-100 (en-ko) from the Hugging Face Hub ...")
    ds = load_dataset("Helsinki-NLP/opus-100", "en-ko", split="train")

    pairs = []
    for ex in ds:
        ko = ex["translation"]["ko"].strip()
        en = ex["translation"]["en"].strip()
        if not ko or not en:
            continue
        if len(ko) > MAX_SENTENCE_CHARS or len(en) > MAX_SENTENCE_CHARS:
            continue
        pairs.append((ko, en))
        if len(pairs) >= num_examples:
            break
    return pairs


try:
    pairs = download_opus100()
    if len(pairs) < 500:
        raise RuntimeError("Downloaded dataset was too small.")
    dataset_source = "OPUS-100 (Helsinki-NLP, via Hugging Face Datasets)"
except Exception as e:
    print(f"Falling back to the bundled sample corpus. Reason: {e}")
    pairs = SAMPLE_PAIRS
    dataset_source = "bundled sample corpus"

lines = [f"{ko}\t{en}" for ko, en in pairs]
path_to_file.write_text("\n".join(lines), encoding="utf-8")

print(f"Dataset source : {dataset_source}")
print(f"Sentence pairs : {len(pairs)}")
print(f"Saved to       : {path_to_file.resolve()}")


In [ ]:
def load_data(path):
  """Load a Korean-English tab-separated file into (target, context) arrays.

  Each line in the file is formatted as: `<Korean sentence>\t<English sentence>`.
  `context` is the source language the encoder reads (Korean), and `target`
  is the language the decoder learns to generate (English). Keeping the
  `target, context` return order matches the rest of this notebook, which
  was originally written for the Spanish -> English tutorial.
  """
  text = path.read_text(encoding='utf-8')

  lines = [line for line in text.splitlines() if line.strip()]
  pairs = [line.split('\t') for line in lines]
  pairs = [p for p in pairs if len(p) == 2]  # drop any malformed lines

  context = np.array([korean for korean, english in pairs])   # Korean (source)
  target = np.array([english for korean, english in pairs])   # English (target)

  return target, context


In [ ]:
target_raw, context_raw = load_data(path_to_file)
print('Korean (context):', context_raw[-1])


In [ ]:
print('English (target):', target_raw[-1])

### Create a tf.data dataset

From these arrays of strings you can create a `tf.data.Dataset` of strings that shuffles and batches them efficiently:

In [ ]:
BUFFER_SIZE = len(context_raw)
BATCH_SIZE = 64

is_train = np.random.uniform(size=(len(target_raw),)) < 0.8

train_raw = (
    tf.data.Dataset
    .from_tensor_slices((context_raw[is_train], target_raw[is_train]))
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE))
val_raw = (
    tf.data.Dataset
    .from_tensor_slices((context_raw[~is_train], target_raw[~is_train]))
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE))

In [ ]:
for example_context_strings, example_target_strings in train_raw.take(1):
  print(example_context_strings[:5])
  print()
  print(example_target_strings[:5])
  break

### Text preprocessing

One of the goals of this tutorial is to build a model that can be exported as a `tf.saved_model`. To make that exported model useful it should take `tf.string` inputs, and return `tf.string` outputs: All the text processing happens inside the model. Mainly using a `layers.TextVectorization` layer.

#### Standardization

The model deals with multilingual text (Korean in, English out) with a
limited vocabulary, so it is important to standardize the input text.

Unlike Spanish or French, Korean (Hangul) has **no accented Latin
characters** and **no case** (no upper/lower case distinction), so the
standardization function needs different logic:

* We use **NFKC** Unicode normalization (instead of NFKD) to fold
  compatibility characters and normalize any full-width punctuation that
  can appear in Korean web text, without decomposing Hangul syllables into
  invalid pieces.
* We **keep every Hangul syllable and Jamo character** — nothing is
  stripped from the Korean alphabet.
* We still lowercase the string (harmless for Hangul, useful for any
  incidental Latin letters/numbers mixed into a sentence).
* We keep a small set of punctuation marks and separate them from words
  with spaces, so the tokenizer treats them as individual tokens.

The `tensorflow_text` package contains a Unicode-normalize operation:

In [ ]:
example_text = tf.constant('그는 오늘 도서관에서 책을 읽었습니다.')

print(example_text.numpy())
print(tf_text.normalize_utf8(example_text, 'NFKC').numpy())


Unicode normalization is the first step of the text-standardization function below:

In [ ]:
def tf_lower_and_split_punct(text):
  # Normalize compatibility characters / full-width punctuation, without
  # destructively decomposing Hangul syllables.
  text = tf_text.normalize_utf8(text, 'NFKC')
  text = tf.strings.lower(text)
  # Keep: whitespace, Hangul syllables (U+AC00-U+D7A3), Hangul Jamo
  # consonants/vowels (U+3131-U+3163), digits, basic Latin letters, and a
  # small set of punctuation marks. Everything else (emoji, stray symbols,
  # ...) is dropped, but no Korean character is ever removed.
  #
  # NOTE: TensorFlow's regex_replace uses RE2, which does not support
  # \uXXXX escapes, so the Hangul ranges are written as literal characters.
  text = tf.strings.regex_replace(text, r'[^ 0-9a-z가-힣ㄱ-ㅎㅏ-ㅣ.?!,]', '')
  # Add spaces around punctuation so it becomes its own token.
  text = tf.strings.regex_replace(text, r'[.?!,]', r' \0 ')
  # Collapse repeated whitespace and strip leading/trailing spaces.
  text = tf.strings.regex_replace(text, r'\s+', ' ')
  text = tf.strings.strip(text)

  text = tf.strings.join(['[START]', text, '[END]'], separator=' ')
  return text


In [ ]:
print(example_text.numpy().decode())
print(tf_lower_and_split_punct(example_text).numpy().decode())


#### Text Vectorization

This standardization function will be wrapped up in a `tf.keras.layers.TextVectorization` layer which will handle the vocabulary extraction and conversion of input text to sequences of tokens.

**Choosing the vocabulary size.** Korean is an agglutinative language:
particles and endings attach directly to word stems (e.g. 학교에서,
학교와, 학교는 are all built from 학교 = "school"). At the whitespace-level
("eojeol") tokenization used here, this produces more distinct surface
forms per root word than English does, so the Korean vocabulary is given a
larger cap (`8,000` tokens) than the English vocabulary (`5,000` tokens,
matching the original tutorial) to capture more of that morphological
variety. In a production system this is exactly the kind of problem
subword tokenizers such as **SentencePiece** or **BPE** solve much more
efficiently — see *Future Improvements* at the end of this notebook.

In [ ]:
context_vocab_size = 8000   # Korean (source) vocabulary size
target_vocab_size = 5000    # English (target) vocabulary size

context_text_processor = tf.keras.layers.TextVectorization(
    standardize=tf_lower_and_split_punct,
    max_tokens=context_vocab_size,
    ragged=True)


The `TextVectorization` layer and many other [Keras preprocessing layers](https://www.tensorflow.org/guide/keras/preprocessing_layers) have an `adapt` method. This method reads one epoch of the training data, and works a lot like `Model.fit`. This `adapt` method initializes the layer based on the data. Here it determines the vocabulary:

In [ ]:
context_text_processor.adapt(train_raw.map(lambda context, target: context))

# Here are the first 10 tokens from the Korean vocabulary:
context_text_processor.get_vocabulary()[:10]


That's the Korean `TextVectorization` layer, now build and `.adapt()` the English one:

In [ ]:
target_text_processor = tf.keras.layers.TextVectorization(
    standardize=tf_lower_and_split_punct,
    max_tokens=target_vocab_size,
    ragged=True)

target_text_processor.adapt(train_raw.map(lambda context, target: target))
target_text_processor.get_vocabulary()[:10]


Now these layers can convert a batch of strings into a batch of token IDs:

In [ ]:
example_tokens = context_text_processor(example_context_strings)
example_tokens[:3, :]

The `get_vocabulary` method can be used to convert token IDs back to text:

In [ ]:
context_vocab = np.array(context_text_processor.get_vocabulary())
tokens = context_vocab[example_tokens[0].numpy()]
' '.join(tokens)

The returned token IDs are zero-padded. This can easily be turned into a mask:

In [ ]:
plt.subplot(1, 2, 1)
plt.pcolormesh(example_tokens.to_tensor())
plt.title('Token IDs')

plt.subplot(1, 2, 2)
plt.pcolormesh(example_tokens.to_tensor() != 0)
plt.title('Mask')

### Process the dataset



The `process_text` function below converts the `Datasets` of strings, into  0-padded tensors of token IDs. It also converts from a `(context, target)` pair to an `((context, target_in), target_out)` pair for training with `keras.Model.fit`. Keras expects `(inputs, labels)` pairs, the inputs are the `(context, target_in)` and the labels are `target_out`. The difference between `target_in` and `target_out` is that they are shifted by one step relative to eachother, so that at each location the label is the next token.

In [ ]:
def process_text(context, target):
  context = context_text_processor(context).to_tensor()
  target = target_text_processor(target)
  targ_in = target[:,:-1].to_tensor()
  targ_out = target[:,1:].to_tensor()
  return (context, targ_in), targ_out


train_ds = train_raw.map(process_text, tf.data.AUTOTUNE)
val_ds = val_raw.map(process_text, tf.data.AUTOTUNE)

Here is the first sequence of each, from the first batch:

In [ ]:
for (ex_context_tok, ex_tar_in), ex_tar_out in train_ds.take(1):
  print(ex_context_tok[0, :10].numpy())
  print()
  print(ex_tar_in[0, :10].numpy())
  print(ex_tar_out[0, :10].numpy())

## The encoder/decoder

The following diagrams shows an overview of the model. In both the encoder is on the left, the decoder is on the right. At each time-step the decoder's output is combined with the encoder's output, to predict the next word.

The original [left] contains a few extra connections that are intentionally omitted from this tutorial's model [right], as they are generally unnecessary, and difficult to implement. Those missing connections are:

1. Feeding the state from the encoder's RNN to the decoder's RNN
2. Feeding the attention output back to the RNN's input.

<table>
<tr>
  <td>
   <img width=500 src="https://www.tensorflow.org/images/seq2seq/attention_mechanism.jpg"/>
  </td>
  <td>
   <img width=380 src="https://www.tensorflow.org/images/tutorials/transformer/RNN+attention.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>The original from <a href=https://arxiv.org/abs/1508.04025v5>Effective Approaches to Attention-based Neural Machine Translation</a></th>
  <th colspan=1>This tutorial's model</th>
<tr>
</table>


Before getting into it define constants for the model:

In [ ]:
UNITS = 256

### The encoder

The goal of the encoder is to process the context sequence into a sequence of vectors that are useful for the decoder as it attempts to predict the next output for each timestep. Since the context sequence is constant, there is no restriction on how information can flow in the encoder, so use a bidirectional-RNN to do the processing:

<table>
<tr>
  <td>
   <img width=500 src="https://tensorflow.org/images/tutorials/transformer/RNN-bidirectional.png"/>
  </td>
</tr>
<tr>
  <th>A bidirectional RNN</th>
<tr>
</table>

The encoder:

1. Takes a list of token IDs (from `context_text_processor`).
3. Looks up an embedding vector for each token (Using a `layers.Embedding`).
4. Processes the embeddings into a new sequence (Using a bidirectional `layers.GRU`).
5. Returns the processed sequence. This will be passed to the attention head.

In [ ]:
class Encoder(tf.keras.layers.Layer):
  def __init__(self, text_processor, units):
    super(Encoder, self).__init__()
    self.text_processor = text_processor
    self.vocab_size = text_processor.vocabulary_size()
    self.units = units

    # The embedding layer converts tokens to vectors
    self.embedding = tf.keras.layers.Embedding(self.vocab_size, units,
                                               mask_zero=True)

    # The RNN layer processes those vectors sequentially.
    self.rnn = tf.keras.layers.Bidirectional(
        merge_mode='sum',
        layer=tf.keras.layers.GRU(units,
                            # Return the sequence and state
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'))

  def call(self, x):
    shape_checker = ShapeChecker()
    shape_checker(x, 'batch s')

    # 2. The embedding layer looks up the embedding vector for each token.
    x = self.embedding(x)
    shape_checker(x, 'batch s units')

    # 3. The GRU processes the sequence of embeddings.
    x = self.rnn(x)
    shape_checker(x, 'batch s units')

    # 4. Returns the new sequence of embeddings.
    return x

  def convert_input(self, texts):
    texts = tf.convert_to_tensor(texts)
    if len(texts.shape) == 0:
      texts = tf.convert_to_tensor(texts)[tf.newaxis]
    context = self.text_processor(texts).to_tensor()
    context = self(context)
    return context

Try it out:

In [ ]:
# Encode the input sequence.
encoder = Encoder(context_text_processor, UNITS)
ex_context = encoder(ex_context_tok)

print(f'Context tokens, shape (batch, s): {ex_context_tok.shape}')
print(f'Encoder output, shape (batch, s, units): {ex_context.shape}')

### The attention layer

The attention layer lets the decoder access the information extracted by the encoder. It computes a vector from the entire context sequence, and adds that to the decoder's output.

The simplest way you could calculate a single vector from the entire sequence would be to take the average across the sequence (`layers.GlobalAveragePooling1D`). An attention layer is similar, but calculates a **weighted** average across the context sequence. Where the weights are calculated from the combination of context and "query" vectors.

<table>
<tr>
  <td>
   <img width=500 src="https://www.tensorflow.org/images/tutorials/transformer/CrossAttention-new-full.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>The attention layer</th>
<tr>
</table>

In [ ]:
class CrossAttention(tf.keras.layers.Layer):
  def __init__(self, units, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(key_dim=units, num_heads=1, **kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x, context):
    shape_checker = ShapeChecker()

    shape_checker(x, 'batch t units')
    shape_checker(context, 'batch s units')

    attn_output, attn_scores = self.mha(
        query=x,
        value=context,
        return_attention_scores=True)

    shape_checker(x, 'batch t units')
    shape_checker(attn_scores, 'batch heads t s')

    # Cache the attention scores for plotting later.
    attn_scores = tf.reduce_mean(attn_scores, axis=1)
    shape_checker(attn_scores, 'batch t s')
    self.last_attention_weights = attn_scores

    x = self.add([x, attn_output])
    x = self.layernorm(x)

    return x

In [ ]:
attention_layer = CrossAttention(UNITS)

# Attend to the encoded tokens
embed = tf.keras.layers.Embedding(target_text_processor.vocabulary_size(),
                                  output_dim=UNITS, mask_zero=True)
ex_tar_embed = embed(ex_tar_in)

result = attention_layer(ex_tar_embed, ex_context)

print(f'Context sequence, shape (batch, s, units): {ex_context.shape}')
print(f'Target sequence, shape (batch, t, units): {ex_tar_embed.shape}')
print(f'Attention result, shape (batch, t, units): {result.shape}')
print(f'Attention weights, shape (batch, t, s):    {attention_layer.last_attention_weights.shape}')

The attention weights will sum to `1` over the context sequence, at each location in the target sequence.

In [ ]:
attention_layer.last_attention_weights[0].numpy().sum(axis=-1)



Here are the attention weights across the context sequences at `t=0`:

In [ ]:
attention_weights = attention_layer.last_attention_weights
mask=(ex_context_tok != 0).numpy()

plt.subplot(1, 2, 1)
plt.pcolormesh(mask*attention_weights[:, 0, :])
plt.title('Attention weights')

plt.subplot(1, 2, 2)
plt.pcolormesh(mask)
plt.title('Mask');


Because of the small-random initialization the attention weights are initially all close to `1/(sequence_length)`. The model will learn to make these less uniform as training progresses.

### The decoder

The decoder's job is to generate predictions for the next token at each location in the target sequence.

1. It looks up embeddings for each token in the target sequence.
2. It uses an RNN to process the target sequence, and keep track of what it has generated so far.
3. It uses RNN output as the "query" to the attention layer, when attending to the encoder's output.
4. At each location in the output it predicts the next token.

When training, the model predicts the next word at each location. So it's important that the information only flows in one direction through the model. The decoder uses a unidirectional (not bidirectional) RNN to process the target sequence.

When running inference with this model it produces one word at a time, and those are fed back into the model.

<table>
<tr>
  <td>
   <img width=500 src="https://tensorflow.org/images/tutorials/transformer/RNN.png"/>
  </td>
</tr>
<tr>
  <th>A unidirectional RNN</th>
<tr>
</table>

Here is the `Decoder` class' initializer. The initializer creates all the necessary layers.

In [ ]:
class Decoder(tf.keras.layers.Layer):
  @classmethod
  def add_method(cls, fun):
    setattr(cls, fun.__name__, fun)
    return fun

  def __init__(self, text_processor, units):
    super(Decoder, self).__init__()
    self.text_processor = text_processor
    self.vocab_size = text_processor.vocabulary_size()
    self.word_to_id = tf.keras.layers.StringLookup(
        vocabulary=text_processor.get_vocabulary(),
        mask_token='', oov_token='[UNK]')
    self.id_to_word = tf.keras.layers.StringLookup(
        vocabulary=text_processor.get_vocabulary(),
        mask_token='', oov_token='[UNK]',
        invert=True)
    self.start_token = self.word_to_id('[START]')
    self.end_token = self.word_to_id('[END]')

    self.units = units


    # 1. The embedding layer converts token IDs to vectors
    self.embedding = tf.keras.layers.Embedding(self.vocab_size,
                                               units, mask_zero=True)

    # 2. The RNN keeps track of what's been generated so far.
    self.rnn = tf.keras.layers.GRU(units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')

    # 3. The RNN output will be the query for the attention layer.
    self.attention = CrossAttention(units)

    # 4. This fully connected layer produces the logits for each
    # output token.
    self.output_layer = tf.keras.layers.Dense(self.vocab_size)

#### Training

Next, the `call` method, takes 3 arguments:

* `inputs` -  a `context, x` pair where:
  * `context` - is the context from the encoder's output.
  * `x` - is the target sequence input.
* `state` - Optional, the previous `state` output from the decoder (the internal state of the decoder's RNN). Pass the state from a previous run to continue generating text where you left off.
* `return_state` - [Default: False] - Set this to `True` to return the RNN state.

In [ ]:
@Decoder.add_method
def call(self,
         context, x,
         state=None,
         return_state=False):
  shape_checker = ShapeChecker()
  shape_checker(x, 'batch t')
  shape_checker(context, 'batch s units')

  # 1. Lookup the embeddings
  x = self.embedding(x)
  shape_checker(x, 'batch t units')

  # 2. Process the target sequence.
  x, state = self.rnn(x, initial_state=state)
  shape_checker(x, 'batch t units')

  # 3. Use the RNN output as the query for the attention over the context.
  x = self.attention(x, context)
  self.last_attention_weights = self.attention.last_attention_weights
  shape_checker(x, 'batch t units')
  shape_checker(self.last_attention_weights, 'batch t s')

  # Step 4. Generate logit predictions for the next token.
  logits = self.output_layer(x)
  shape_checker(logits, 'batch t target_vocab_size')

  if return_state:
    return logits, state
  else:
    return logits

That will be sufficient for training. Create an instance of the decoder to test out:

In [ ]:
decoder = Decoder(target_text_processor, UNITS)

In training you'll use the decoder like this:

Given the context and target tokens, for each target token it predicts the next target token.

In [ ]:
logits = decoder(ex_context, ex_tar_in)

print(f'encoder output shape: (batch, s, units) {ex_context.shape}')
print(f'input target tokens shape: (batch, t) {ex_tar_in.shape}')
print(f'logits shape shape: (batch, target_vocabulary_size) {logits.shape}')

#### Inference

To use it for inference you'll need a couple more methods:

In [ ]:
@Decoder.add_method
def get_initial_state(self, context):
  batch_size = tf.shape(context)[0]
  start_tokens = tf.fill([batch_size, 1], self.start_token)
  done = tf.zeros([batch_size, 1], dtype=tf.bool)
  embedded = self.embedding(start_tokens)
  initial_state = tf.zeros([batch_size, self.units], dtype=embedded.dtype)
  return start_tokens, done, initial_state


In [ ]:
@Decoder.add_method
def tokens_to_text(self, tokens):
  words = self.id_to_word(tokens)
  result = tf.strings.reduce_join(words, axis=-1, separator=' ')
  result = tf.strings.regex_replace(result, '^ *\[START\] *', '')
  result = tf.strings.regex_replace(result, ' *\[END\] *$', '')
  return result

In [ ]:
@Decoder.add_method
def get_next_token(self, context, next_token, done, state, temperature = 0.0):
  logits, state = self(
    context, next_token,
    state = state,
    return_state=True)

  if temperature == 0.0:
    next_token = tf.argmax(logits, axis=-1)
  else:
    logits = logits[:, -1, :]/temperature
    next_token = tf.random.categorical(logits, num_samples=1)

  # If a sequence produces an `end_token`, set it `done`
  done = done | (next_token == self.end_token)
  # Once a sequence is done it only produces 0-padding.
  next_token = tf.where(done, tf.constant(0, dtype=tf.int64), next_token)

  return next_token, done, state

With those extra functions, you can write a generation loop:

In [ ]:
# Setup the loop variables.
next_token, done, state = decoder.get_initial_state(ex_context)
tokens = []

for n in range(10):
  # Run one step.
  next_token, done, state = decoder.get_next_token(
      ex_context, next_token, done, state, temperature=1.0)
  # Add the token to the output.
  tokens.append(next_token)

# Stack all the tokens together.
tokens = tf.concat(tokens, axis=-1) # (batch, t)

# Convert the tokens back to a a string
result = decoder.tokens_to_text(tokens)
result[:3].numpy()

Since the model's untrained, it outputs items from the vocabulary almost uniformly at random.

## The model

Now that you have all the model components, combine them to build the model for training:

In [ ]:
class Translator(tf.keras.Model):
  @classmethod
  def add_method(cls, fun):
    setattr(cls, fun.__name__, fun)
    return fun

  def __init__(self, units,
               context_text_processor,
               target_text_processor):
    super().__init__()
    # Build the encoder and decoder
    encoder = Encoder(context_text_processor, units)
    decoder = Decoder(target_text_processor, units)

    self.encoder = encoder
    self.decoder = decoder

  def call(self, inputs):
    context, x = inputs
    context = self.encoder(context)
    logits = self.decoder(context, x)

    #TODO(b/250038731): remove this
    try:
      # Delete the keras mask, so keras doesn't scale the loss+accuracy.
      del logits._keras_mask
    except AttributeError:
      pass

    return logits

During training the model will be used like this:

In [ ]:
model = Translator(UNITS, context_text_processor, target_text_processor)

logits = model((ex_context_tok, ex_tar_in))

print(f'Context tokens, shape: (batch, s, units) {ex_context_tok.shape}')
print(f'Target tokens, shape: (batch, t) {ex_tar_in.shape}')
print(f'logits, shape: (batch, t, target_vocabulary_size) {logits.shape}')

### Train

For training, you'll want to implement your own masked loss and accuracy functions:

In [ ]:
def masked_loss(y_true, y_pred):
    # Calculate the loss for each item in the batch.
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction='none')
    loss = loss_fn(y_true, y_pred)

    # Mask off the losses on padding.
    mask = tf.cast(y_true != 0, loss.dtype)
    loss *= mask

    # Return the total.
    return tf.reduce_sum(loss)/tf.reduce_sum(mask)

In [ ]:
def masked_acc(y_true, y_pred):
    # Calculate the loss for each item in the batch.
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.cast(y_pred, y_true.dtype)

    match = tf.cast(y_true == y_pred, tf.float32)
    mask = tf.cast(y_true != 0, tf.float32)

    return tf.reduce_sum(match)/tf.reduce_sum(mask)

Configure the model for training:

In [ ]:
model.compile(optimizer='adam',
              loss=masked_loss,
              metrics=[masked_acc, masked_loss])

The model is randomly initialized, and should give roughly uniform output probabilities. So it's easy to predict what the initial values of the metrics should be:

In [ ]:
vocab_size = 1.0 * target_text_processor.vocabulary_size()

{"expected_loss": tf.math.log(vocab_size).numpy(),
 "expected_acc": 1/vocab_size}

That should roughly match the values returned by running a few steps of evaluation:

In [ ]:
model.evaluate(val_ds, steps=20, return_dict=True)

To keep the original training strategy (Adam optimizer, masked
cross-entropy loss, `EarlyStopping`) while making training a bit more
robust, we add two more standard Keras callbacks:

* **`ModelCheckpoint`** — saves the best weights seen during training
  (by validation loss), so we can always reload the best checkpoint.
* **`ReduceLROnPlateau`** — lowers the learning rate when validation loss
  stops improving, which often helps squeeze out a bit more accuracy once
  training plateaus.

The overall architecture and training strategy are unchanged.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_kor_eng_model.weights.h5',
        save_best_only=True,
        save_weights_only=True,
        monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]

train_start_time = time.time()

history = model.fit(
    train_ds.repeat(),
    epochs=100,
    steps_per_epoch=100,
    validation_data=val_ds,
    validation_steps=20,
    callbacks=callbacks)

train_end_time = time.time()
training_time_seconds = train_end_time - train_start_time
print(f'Training took {training_time_seconds / 60:.1f} minutes '
      f'({len(history.history["loss"])} epochs).')


In [ ]:
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.ylim([0, max(plt.ylim())])
plt.xlabel('Epoch #')
plt.ylabel('CE/token')
plt.legend()

In [ ]:
plt.plot(history.history['masked_acc'], label='accuracy')
plt.plot(history.history['val_masked_acc'], label='val_accuracy')
plt.ylim([0, max(plt.ylim())])
plt.xlabel('Epoch #')
plt.ylabel('CE/token')
plt.legend()

### Translate

Now that the model is trained, implement a function to execute the full
Korean `=>` English translation. This code is basically identical to the
[inference example](#inference) in the [decoder section](#the_decoder),
but it also captures the attention weights so we can visualize them.

In [ ]:
#@title
@Translator.add_method
def translate(self,
              texts, *,
              max_length=50,
              temperature=0.0):
  # Process the input texts
  context = self.encoder.convert_input(texts)
  batch_size = tf.shape(texts)[0]

  # Setup the loop inputs
  tokens = []
  attention_weights = []
  next_token, done, state = self.decoder.get_initial_state(context)

  for _ in range(max_length):
    # Generate the next token
    next_token, done, state = self.decoder.get_next_token(
        context, next_token, done,  state, temperature)

    # Collect the generated tokens
    tokens.append(next_token)
    attention_weights.append(self.decoder.last_attention_weights)

    if tf.executing_eagerly() and tf.reduce_all(done):
      break

  # Stack the lists of tokens and attention weights.
  tokens = tf.concat(tokens, axis=-1)   # t*[(batch 1)] -> (batch, t)
  self.last_attention_weights = tf.concat(attention_weights, axis=1)  # t*[(batch 1 s)] -> (batch, t s)

  result = self.decoder.tokens_to_text(tokens)
  return result

Here are the two helper methods, used above, to convert tokens to text, and to get the next token:

In [ ]:
result = model.translate(['오늘 날씨가 정말 좋네요.'])  # Expected: something like "the weather is really nice today."
result[0].numpy().decode()


Use that to generate the attention plot. `plot_attention` reuses the same
`tf_lower_and_split_punct` standardization defined above, so the tick
labels shown are the Korean tokens the model actually saw, and the
generated English tokens it produced.

In [ ]:
#@title
@Translator.add_method
def plot_attention(self, text, **kwargs):
  assert isinstance(text, str)
  output = self.translate([text], **kwargs)
  output = output[0].numpy().decode()

  attention = self.last_attention_weights[0]

  context = tf_lower_and_split_punct(text)
  context = context.numpy().decode().split()

  output = tf_lower_and_split_punct(output)
  output = output.numpy().decode().split()[1:]

  fig = plt.figure(figsize=(10, 10))
  ax = fig.add_subplot(1, 1, 1)

  ax.matshow(attention, cmap='viridis', vmin=0.0)

  fontdict = {'fontsize': 14}

  ax.set_xticklabels([''] + context, fontdict=fontdict, rotation=90)
  ax.set_yticklabels([''] + output, fontdict=fontdict)

  ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
  ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

  ax.set_xlabel('Input text')
  ax.set_ylabel('Output text')

In [ ]:
model.plot_attention('오늘 날씨가 정말 좋네요.')  # "the weather is really nice today."

**How to interpret this heatmap.** Each row corresponds to one generated
English word (read top to bottom, in generation order); each column
corresponds to one Korean input token (read left to right, including the
`[START]`/`[END]` markers). A brighter cell at `(row, column)` means the
decoder relied more heavily on that Korean token when producing that
English word. In a well-trained model you'll typically see a roughly
diagonal band — English words tend to align with Korean tokens in a
similar relative position — with occasional off-diagonal attention where
Korean's different word order (e.g. verb-final sentence structure) requires
the decoder to "look back" or "look ahead" in the source sentence.

Translate a few more sentences and plot them:

In [ ]:
%%time
# Expected: something like "I am studying artificial intelligence."
model.plot_attention('저는 인공지능을 공부하고 있습니다.')


In [ ]:
%%time
# Expected: something like "thank you for helping me."
model.plot_attention('도와주셔서 감사합니다.')


The short sentences often work well, but if the input is too long the model literally loses focus and stops providing reasonable predictions. There are two main reasons for this:

1. The model was trained with teacher-forcing feeding the correct token at each step, regardless of the model's predictions. The model could be made more robust if it were sometimes fed its own predictions.
2. The model only has access to its previous output through the RNN state. If the RNN state looses track of where it was in the context sequence there's no way for the model to recover. [Transformers](transformer.ipynb) improve on this by letting the decoder look at what it has output so far.

The raw data is sorted by length, so try translating the longest sequence:

In [ ]:
long_text = context_raw[-1]

import textwrap
print('Expected output:\n', '\n'.join(textwrap.wrap(target_raw[-1])))

In [ ]:
model.plot_attention(long_text)

The `translate` function works on batches, so if you have multiple texts to translate you can pass them all at once, which is much more efficient than translating them one at a time:

In [ ]:
inputs = [
    '이 근처에 은행이 있어요?',       # "Is there a bank near here?"
    '저는 김치를 매우 좋아합니다.',   # "I like kimchi very much."
    '그는 대학원에서 인공지능을 연구합니다.',  # "He researches artificial intelligence in graduate school."
]


In [ ]:
%%time
for t in inputs:
  print(model.translate([t])[0].numpy().decode())

print()

In [ ]:
%%time
result = model.translate(inputs)

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

So overall this text generation function mostly gets the job done, but so you've only used it here in python with eager execution. Let's try to export it next:

## Evaluation: BLEU Score

So far we've only eyeballed a handful of translations. To get a more
objective measure of translation quality, we compute the **BLEU
(Bilingual Evaluation Understudy) score** on a sample of held-out
validation sentences.

**What BLEU measures.** BLEU compares a model's predicted translation
against one or more human reference translations by counting how many
n-grams (contiguous sequences of 1 to 4 words, typically) in the
prediction also appear in the reference, then combines these n-gram
precisions into a single score. A **brevity penalty** discourages the
model from scoring well by generating overly short outputs. BLEU ranges
from 0 to 100 (or 0 to 1, depending on the implementation); higher is
better. It is not a perfect measure of translation quality — it doesn't
understand meaning or synonyms — but it is a widely used, reproducible
benchmark that makes it easy to compare models.

In [ ]:
import sacrebleu


def evaluate_bleu(translator, source_sentences, reference_sentences,
                   n_samples=200, batch_size=32, seed=42):
  """Translate a random sample of held-out sentences and compute corpus BLEU."""
  rng = np.random.default_rng(seed)
  n_samples = min(n_samples, len(source_sentences))
  sample_idx = rng.choice(len(source_sentences), size=n_samples, replace=False)

  sources = [source_sentences[i] for i in sample_idx]
  references = [reference_sentences[i] for i in sample_idx]
  predictions = []

  for start in range(0, len(sources), batch_size):
    batch = sources[start:start + batch_size]
    translated = translator.translate(batch)
    predictions.extend(t.numpy().decode() for t in translated)

  bleu = sacrebleu.corpus_bleu(predictions, [references])
  return bleu.score, sources, references, predictions


val_context = context_raw[~is_train]
val_target = target_raw[~is_train]

bleu_score, eval_sources, eval_references, eval_predictions = evaluate_bleu(
    model, val_context, val_target, n_samples=200)

print(f'Corpus BLEU score on {len(eval_sources)} held-out sentences: {bleu_score:.2f}')


Here are a few sample translations from that evaluation set, showing the Korean input, the ground-truth English reference, and the model's predicted translation:

In [ ]:
for i in range(5):
  print(f'Korean input : {eval_sources[i]}')
  print(f'Reference    : {eval_references[i]}')
  print(f'Predicted    : {eval_predictions[i]}')
  print('-' * 60)


## Testing Section

To get a feel for how translation quality changes with sentence
complexity, here are three test sentences of increasing difficulty: a
short greeting, a medium-length descriptive sentence, and a longer
sentence with more complex grammar and an abstract topic.

In [ ]:
test_cases = [
    ('Easy',   '안녕하세요.',                     'Hello.'),
    ('Medium', '오늘 날씨가 정말 좋네요.',       'The weather is really nice today.'),
    ('Hard',   '저는 인공지능을 공부하고 있습니다.', 'I am studying artificial intelligence.'),
]

for level, korean_text, reference in test_cases:
  predicted = model.translate([korean_text])[0].numpy().decode()
  print(f'[{level}]')
  print(f'  Input (Korean)            : {korean_text}')
  print(f'  \u2193')
  print(f'  Predicted English         : {predicted}')
  print(f'  Reference Translation     : {reference}')
  print()


You can also visualize the attention pattern for each of these test cases:

In [ ]:
for level, korean_text, _ in test_cases:
  print(level)
  model.plot_attention(korean_text)


### Export

If you want to export this model you'll need to wrap the `translate` method in a `tf.function`. That implementation will get the job done:


In [ ]:
class Export(tf.Module):
  def __init__(self, model):
    self.model = model

  @tf.function(input_signature=[tf.TensorSpec(dtype=tf.string, shape=[None])])
  def translate(self, inputs):
    return self.model.translate(inputs)

In [ ]:
export = Export(model)

Run the `tf.function` once to compile it:

In [ ]:
%%time
_ = export.translate(tf.constant(inputs))

In [ ]:
%%time
result = export.translate(tf.constant(inputs))

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

Now that the function has been traced it can be exported using `saved_model.save`:

In [ ]:
%%time
tf.saved_model.save(export, 'translator',
                    signatures={'serving_default': export.translate})

In [ ]:
%%time
reloaded = tf.saved_model.load('translator')
_ = reloaded.translate(tf.constant(inputs)) #warmup

In [ ]:
%%time
result = reloaded.translate(tf.constant(inputs))

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

#### [Optional] Use a dynamic loop

It's worth noting that this initial implementation is not optimal. It uses a python loop:

```
for _ in range(max_length):
  ...
  if tf.executing_eagerly() and tf.reduce_all(done):
    break
```

The python loop is relatively simple but when `tf.function` converts this to a graph, it **statically unrolls** that loop. Unrolling the loop has two disadvantages:

1. It makes `max_length` copies of the loop body. So the generated graphs take longer to build, save and load.
1. You have to choose a fixed value for the `max_length`.
1. You can't `break` from a statically unrolled loop. The `tf.function`
  version will run the full `max_length` iterations on every call.
  That's why the `break` only works with eager execution. This is
  still marginally faster than eager execution, but not as fast as it could be.


To fix these shortcomings, the `translate_dynamic` method, below, uses a tensorflow loop:

```
for t in tf.range(max_length):
  ...
  if tf.reduce_all(done):
      break
```

It looks like a python loop, but when you use a tensor as the input to a `for` loop (or the condition of a `while` loop) `tf.function` converts it to a dynamic loop using operations like `tf.while_loop`.

There's no need for a `max_length` here it's just in case the model gets stuck generating a loop like: `the united states of the united states of the united states...`.

On the down side, to accumulate tokens from this dynamic loop you can't just append them to a python `list`, you need to use a `tf.TensorArray`:

```
tokens = tf.TensorArray(tf.int64, size=1, dynamic_size=True)
...
for t in tf.range(max_length):
    ...
    tokens = tokens.write(t, next_token) # next_token shape is (batch, 1)
  ...
  tokens = tokens.stack()
  tokens = einops.rearrange(tokens, 't batch 1 -> batch t')
```

This version of the code can be quite a bit more efficient:

In [ ]:
#@title
@Translator.add_method
def translate(self,
              texts,
              *,
              max_length=500,
              temperature=tf.constant(0.0)):
  shape_checker = ShapeChecker()
  context = self.encoder.convert_input(texts)
  batch_size = tf.shape(context)[0]
  shape_checker(context, 'batch s units')

  next_token, done, state = self.decoder.get_initial_state(context)

  # initialize the accumulator
  tokens = tf.TensorArray(tf.int64, size=1, dynamic_size=True)

  for t in tf.range(max_length):
    # Generate the next token
    next_token, done, state = self.decoder.get_next_token(
        context, next_token, done, state, temperature)
    shape_checker(next_token, 'batch t1')

    # Collect the generated tokens
    tokens = tokens.write(t, next_token)

    # if all the sequences are done, break
    if tf.reduce_all(done):
      break

  # Convert the list of generated token ids to a list of strings.
  tokens = tokens.stack()
  shape_checker(tokens, 't batch t1')
  tokens = einops.rearrange(tokens, 't batch 1 -> batch t')
  shape_checker(tokens, 'batch t')

  text = self.decoder.tokens_to_text(tokens)
  shape_checker(text, 'batch')

  return text

With eager execution this implementation performs on par with the original:

In [ ]:
%%time
result = model.translate(inputs)

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

But when you wrap it in a `tf.function` you'll notice two differences.

In [ ]:
class Export(tf.Module):
  def __init__(self, model):
    self.model = model

  @tf.function(input_signature=[tf.TensorSpec(dtype=tf.string, shape=[None])])
  def translate(self, inputs):
    return self.model.translate(inputs)

In [ ]:
export = Export(model)

First, it's much quicker to trace, since it only creates one copy of the loop body:

In [ ]:
%%time
_ = export.translate(inputs)

The `tf.function` is much faster than running with eager execution, and on small inputs it's often several times faster than the unrolled version, because it can break out of the loop.

In [ ]:
%%time
result = export.translate(inputs)

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

So save this version as well:

In [ ]:
%%time
tf.saved_model.save(export, 'dynamic_translator',
                    signatures={'serving_default': export.translate})

In [ ]:
%%time
reloaded = tf.saved_model.load('dynamic_translator')
_ = reloaded.translate(tf.constant(inputs)) #warmup

In [ ]:
%%time
result = reloaded.translate(tf.constant(inputs))

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

## Results Summary

Run the cell below after training to print a compact summary of this run.
It pulls together the dataset size, vocabulary sizes, training time, final
loss/accuracy, BLEU score, and a couple of example translations — useful
for a portfolio README or a model card.

In [ ]:
print('=' * 60)
print('Korean -> English NMT with Attention: Results Summary')
print('=' * 60)
print(f'Dataset source          : {dataset_source}')
print(f'Sentence pairs (total)  : {len(pairs)}')
print(f'Training pairs          : {int(is_train.sum())}')
print(f'Validation pairs        : {int((~is_train).sum())}')
print(f'Korean vocabulary size  : {context_text_processor.vocabulary_size()}')
print(f'English vocabulary size : {target_text_processor.vocabulary_size()}')
print(f'Training time           : {training_time_seconds / 60:.1f} minutes '
      f'({len(history.history["loss"])} epochs)')
print(f'Final training loss     : {history.history["loss"][-1]:.4f}')
print(f'Final validation loss   : {history.history["val_loss"][-1]:.4f}')
print(f'Final training accuracy : {history.history["masked_acc"][-1]:.4f}')
print(f'Final val. accuracy     : {history.history["val_masked_acc"][-1]:.4f}')
print(f'BLEU score ({len(eval_sources)} val. samples): {bleu_score:.2f}')
print()
print('Example translations:')
for level, korean_text, reference in test_cases:
  predicted = model.translate([korean_text])[0].numpy().decode()
  print(f'  [{level}] {korean_text}')
  print(f'        -> {predicted}   (reference: {reference})')
print('=' * 60)


## Future Improvements

This notebook intentionally keeps the original RNN + attention
architecture from the TensorFlow tutorial, trained on a modest number of
sentence pairs, so it can run end-to-end in Google Colab in a reasonable
amount of time. Here are natural next steps to push translation quality
further, roughly in order of impact:

1. **Transformer architecture.** Replace the GRU-based encoder/decoder
   with self-attention Transformer blocks, which parallelize training and
   typically capture longer-range dependencies better than RNNs —
   especially useful for Korean's flexible word order.
2. **Multi-head attention.** Even within an RNN-based model, extending the
   single-head `CrossAttention` layer to multiple heads (as in
   `layers.MultiHeadAttention` with `num_heads > 1`) lets the model attend
   to different aspects of the context simultaneously.
3. **SentencePiece / BPE tokenization.** Word-level (eojeol) tokenization
   struggles with Korean's rich morphology (see the vocabulary-size
   discussion above). A subword tokenizer trained jointly on Korean and
   English text would shrink the vocabulary, reduce out-of-vocabulary
   tokens, and usually improve BLEU.
4. **Beam search decoding.** The `translate` method here uses greedy
   (`argmax`) decoding. Beam search keeps several candidate translations
   at each step and often produces noticeably more fluent output,
   especially for longer sentences.
5. **Larger, more diverse datasets.** Training on the full OPUS-100
   Korean-English split (or additional corpora such as AI Hub's
   Korean-English translation corpus, or the Korean-English news
   commentary corpus) instead of a capped subset would expose the model
   to much more vocabulary and sentence structure.
6. **Fine-tuning pretrained multilingual models.** Rather than training
   from scratch, fine-tuning a pretrained multilingual sequence-to-sequence
   model (e.g. mBART, mT5, or NLLB) on Korean-English data typically gets
   to a much higher BLEU score with far less training data and time.

## Next Steps (from the original tutorial)

* [Download a different dataset](http://www.manythings.org/anki/) to
  experiment with other language pairs.
* Try the
  [transformer tutorial](https://www.tensorflow.org/text/tutorials/transformer)
  which implements a similar translation task but uses Transformer layers
  instead of RNNs, along with subword tokenization.
* Visit the
  [`tensorflow_addons.seq2seq` tutorial](https://www.tensorflow.org/addons/tutorials/networks_seq2seq_nmt),
  which demonstrates higher-level functionality for this kind of
  sequence-to-sequence model, such as `seq2seq.BeamSearchDecoder`.